# Cross-Cell Evaluation of TE-Q-Transformer on the CALCE Battery Dataset
## Physics-Guided Quantum-Classical SOH Estimation on Prismatic $\mathrm{LiCoO}_2$ Cells

**Dataset:** CALCE CS2 Prismatic Battery Series (`CS2_35`, `CS2_36`, `CS2_37`, `CS2_38`)  
**Form Factor & Chemistry:** Prismatic geometry, Lithium Cobalt Oxide ($\mathrm{LiCoO}_2$)  
**Primary Benchmark Context:** The NASA Ames cylindrical benchmark remains the primary model-to-model baseline comparison suite. This notebook presents a dedicated cross-cell and temporal extrapolation evaluation of the proposed **TE-Q-Transformer** without baseline models.

---

### Experimental Protocol & Canonical Partition:
- **Training Partition:** Complete cycles of `CS2_35` and `CS2_36`, plus the first 70% of `CS2_37` (2,309 cycles total).
- **Testing Partition:**
  1. **Cross-Cell Unseen Evaluation:** Complete held-out `CS2_38` cell (966 cycles) with unseen manufacturing and degradation dynamics.
  2. **Temporal Extrapolation Evaluation:** Remaining 30% of `CS2_37` (`CS2_37_test`, 275 cycles) under extended cycling.
  3. **Total Test Cycles:** 1,241 cycles.
- **Physics-Guided Normalization:** Min-max scaling fitted strictly on the training partition for `Voltage`, `Current`, and `Time_norm`. `Temperature` is preserved in raw Celsius for Arrhenius kinetic equations.
- **Architecture:** TE-Q-Transformer with 4-qubit Arrhenius quantum embedding, Conv1D smoother, and 3-layer Transformer encoder (92,554 parameters).


In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os
import sys
import time
import math
import json
import random
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple

# Ensure PennyLane is installed
try:
    import pennylane as qml
except ImportError:
    print("[Setup] Installing PennyLane...")
    os.system(f"{sys.executable} -m pip install -q pennylane")
    import pennylane as qml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def seed_everything(seed: int = 42) -> None:
    """Sets deterministic random seeds across all libraries."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print(f"[Hardware] Compute Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"[Hardware] GPU Name       : {torch.cuda.get_device_name(0)}")
    print(f"[Hardware] GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"[Hardware] PyTorch Version: {torch.__version__}")
print(f"[Hardware] PennyLane Ver  : {qml.__version__}")
print("=" * 70)


In [ ]:
# ==============================================================================
# 2. DATASET PATH RESOLUTION & OUTPUT DIRECTORIES
# ==============================================================================
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working") / "outputs" / "calce"
elif Path("/content").exists():
    OUTPUT_DIR = Path("/content") / "outputs" / "calce"
else:
    OUTPUT_DIR = Path("outputs/calce")
FIG_DIR = OUTPUT_DIR / "figures"
RESULTS_DIR = OUTPUT_DIR / "results"
CKPT_DIR = OUTPUT_DIR / "checkpoints"

for d in [OUTPUT_DIR, FIG_DIR, RESULTS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def resolve_calce_data_dir() -> Path:
    """Locates the canonical preprocessed CALCE data across local and Kaggle paths."""
    candidates = [
        Path("datasets/CALCE/processed"),
        Path("/kaggle/working/TE-Q-Transformer/datasets/CALCE/processed"),
        Path("/kaggle/working/datasets/CALCE/processed"),
        Path("../../datasets/CALCE/processed"),
        Path("../datasets/CALCE/processed"),
        Path("/kaggle/input/datasets/zadidallisan/calce-data/calce"),
        Path("/kaggle/input/calce-data/calce"),
        Path("/kaggle/input/calce-data"),
        Path("calce/calce/extracted_teq"),
        Path("Dataset/calce"),
    ]
    for candidate in candidates:
        if candidate.exists() and (candidate / "CS2_38_soh.npy").exists():
            print(f"[Data] Found CALCE dataset directory: {candidate.resolve()}")
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate processed CALCE dataset files.\n"
        "Expected CS2_35, CS2_36, CS2_37, CS2_38 (.npy) files in datasets/CALCE/processed/"
    )

CALCE_DATA_DIR = resolve_calce_data_dir()


In [ ]:
# ==============================================================================
# 3. LOCKED CALCE DATA LOADING, SCALING, & PARTITIONING
# ==============================================================================
CALCE_FULL_TRAIN_CELLS = ("CS2_35", "CS2_36")
CALCE_FULL_TEST_CELLS  = ("CS2_38",)
CALCE_SPLIT_CELL_ID    = "CS2_37"
CALCE_SPLIT_RATIO      = 0.70  # First 70% train, remaining 30% test
FEATURE_IDX_TO_SCALE   = (0, 1, 3)  # Voltage(0), Current(1), Time_norm(3); Temp(2) left raw in Celsius
SEQUENCE_LENGTH        = 512

class CALCEBatteryDataset(Dataset):
    """PyTorch Dataset for CALCE discharge windows and SOH labels."""
    def __init__(self, X: torch.Tensor, y: torch.Tensor) -> None:
        self.X = X.float()
        self.y = y.float()
    def __len__(self) -> int:
        return len(self.y)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

def load_cell_arrays(data_dir: Path, cell_id: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Loads unscaled X sequences, SOH targets, and cycle indices."""
    x_path = data_dir / f"{cell_id}_X_unscaled.npy"
    if not x_path.exists():
        x_path = data_dir / f"{cell_id}_X.npy"
    y_path = data_dir / f"{cell_id}_soh.npy"
    c_path = data_dir / f"{cell_id}_cycle.npy"
    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(f"Missing files for {cell_id} in {data_dir}")
    X = np.load(x_path)
    y = np.load(y_path)
    cycles = np.load(c_path) if c_path.exists() else np.arange(1, len(y) + 1)
    return X.astype(np.float32, copy=True), y.astype(np.float32, copy=False), cycles

def normalize_soh_per_cell(y: np.ndarray, cell_id: str) -> np.ndarray:
    """Normalizes SOH as capacity relative to first cycle (C_k / C_0)."""
    c0 = float(y[0])
    if not np.isfinite(c0) or c0 <= 0.0:
        raise ValueError(f"{cell_id} has invalid initial capacity {c0}")
    return (y / np.float32(c0)).astype(np.float32, copy=False)

def load_full_cell(data_dir: Path, cell_id: str) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    X, y, cycles = load_cell_arrays(data_dir, cell_id)
    y = normalize_soh_per_cell(y, cell_id)
    return torch.from_numpy(X).float(), torch.from_numpy(y).float(), cycles

def split_cell_70_30(data_dir: Path, cell_id: str) -> Tuple[Tuple[torch.Tensor, torch.Tensor, np.ndarray], Tuple[torch.Tensor, torch.Tensor, np.ndarray]]:
    X, y, cycles = load_cell_arrays(data_dir, cell_id)
    y = normalize_soh_per_cell(y, cell_id)
    split_idx = int(len(y) * CALCE_SPLIT_RATIO)
    train_part = (torch.from_numpy(X[:split_idx].copy()).float(), torch.from_numpy(y[:split_idx].copy()).float(), cycles[:split_idx])
    test_part  = (torch.from_numpy(X[split_idx:].copy()).float(), torch.from_numpy(y[split_idx:].copy()).float(), cycles[split_idx:])
    return train_part, test_part

def fit_train_scaler(train_X: torch.Tensor | np.ndarray) -> MinMaxScaler:
    """Fits MinMaxScaler exclusively on training partition channels (Voltage, Current, Time_norm)."""
    if isinstance(train_X, torch.Tensor):
        train_X = train_X.cpu().numpy()
    flat_scaled = train_X[:, :, FEATURE_IDX_TO_SCALE].reshape(-1, len(FEATURE_IDX_TO_SCALE))
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(flat_scaled)
    return scaler

def transform_with_scaler(X: torch.Tensor | np.ndarray, scaler: MinMaxScaler, clip: bool = True) -> torch.Tensor:
    """Applies trained scaler to features while preserving Temperature_C in raw Celsius."""
    X_tensor = torch.from_numpy(X.copy()).float() if isinstance(X, np.ndarray) else X.clone().float()
    n_samples, seq_len, _ = X_tensor.shape
    flat_scaled = X_tensor[:, :, FEATURE_IDX_TO_SCALE].reshape(-1, len(FEATURE_IDX_TO_SCALE)).cpu().numpy()
    transformed = scaler.transform(flat_scaled)
    if clip:
        transformed = np.clip(transformed, 0.0, 1.0)
    reshaped = torch.from_numpy(transformed.reshape(n_samples, seq_len, len(FEATURE_IDX_TO_SCALE))).float()
    for i, orig_idx in enumerate(FEATURE_IDX_TO_SCALE):
        X_tensor[:, :, orig_idx] = reshaped[:, :, i]
    return X_tensor

def get_calce_dataloaders(data_dir: Path = CALCE_DATA_DIR, batch_size: int = 8) -> Tuple[DataLoader, Dict[str, DataLoader], Dict[str, np.ndarray], MinMaxScaler]:
    train_X_list, train_y_list = [], []
    for cell_id in CALCE_FULL_TRAIN_CELLS:
        cx, cy, _ = load_full_cell(data_dir, cell_id)
        train_X_list.append(cx)
        train_y_list.append(cy)

    (split_tr_x, split_tr_y, _), (split_te_x, split_te_y, split_te_c) = split_cell_70_30(data_dir, CALCE_SPLIT_CELL_ID)
    train_X_list.append(split_tr_x)
    train_y_list.append(split_tr_y)

    train_X_raw = torch.cat(train_X_list, dim=0)
    train_y = torch.cat(train_y_list, dim=0)

    scaler = fit_train_scaler(train_X_raw)
    train_X = transform_with_scaler(train_X_raw, scaler, clip=True)
    train_ds = CALCEBatteryDataset(train_X, train_y)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    test_loaders: Dict[str, DataLoader] = {}
    test_cycles: Dict[str, np.ndarray] = {}
    for cell_id in CALCE_FULL_TEST_CELLS:
        te_x, te_y, te_c = load_full_cell(data_dir, cell_id)
        te_x_scaled = transform_with_scaler(te_x, scaler, clip=True)
        test_loaders[cell_id] = DataLoader(CALCEBatteryDataset(te_x_scaled, te_y), batch_size=batch_size, shuffle=False)
        test_cycles[cell_id] = te_c

    split_te_x_scaled = transform_with_scaler(split_te_x, scaler, clip=True)
    test_loaders[f"{CALCE_SPLIT_CELL_ID}_test"] = DataLoader(
        CALCEBatteryDataset(split_te_x_scaled, split_te_y), batch_size=batch_size, shuffle=False
    )
    test_cycles[f"{CALCE_SPLIT_CELL_ID}_test"] = split_te_c

    return train_loader, test_loaders, test_cycles, scaler

train_loader, test_loaders, test_cycles, scaler = get_calce_dataloaders(CALCE_DATA_DIR, batch_size=8)
print("=" * 70)
print(f"Training set samples        : {len(train_loader.dataset)} cycles (CS2_35, CS2_36, CS2_37 70%)")
print(f"Held-out unseen cell CS2_38 : {len(test_loaders['CS2_38'].dataset)} cycles (Cross-cell validation)")
print(f"Held-out tail CS2_37_test   : {len(test_loaders['CS2_37_test'].dataset)} cycles (Temporal extrapolation)")
print(f"Total evaluation cycles     : {len(test_loaders['CS2_38'].dataset) + len(test_loaders['CS2_37_test'].dataset)} cycles")
print("=" * 70)


In [ ]:
# ==============================================================================
# 4. PROPOSED MODEL: TE-Q-TRANSFORMER (NO BASELINES)
# ==============================================================================
@dataclass(frozen=True)
class TEQTransformerConfig:
    """Architectural hyperparameters of the proposed TE-Q-Transformer."""
    input_dim: int = 4
    seq_len: int = 512
    quantum_dim: int = 4
    d_model: int = 64
    n_heads: int = 2
    n_layers: int = 3
    dim_feedforward: int = 64
    dropout: float = 0.0
    q_device: str = "default.qubit"
    entangler_layers: int = 1
    entangler_type: str = "rich"
    temporal_kernel_size: int = 3
    head_hidden_dim: int = 64

class PositionalEncodingTEQ(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model: int, max_len: int = 1024) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]

class QuantumEmbeddingLayer(nn.Module):
    """Physics-Guided Arrhenius Temperature Encoding & Parameterized Quantum Circuit."""
    def __init__(self, n_qubits: int = 4, q_device: str = "default.qubit", entangler_layers: int = 1) -> None:
        super().__init__()
        self.n_qubits = n_qubits
        self.dev = qml.device(q_device, wires=n_qubits)
        self.T_ref = nn.Parameter(torch.tensor(298.15), requires_grad=False)
        self.R = nn.Parameter(torch.tensor(8.314), requires_grad=False)
        # Trainable Arrhenius activation energies (initialized from degradation literature)
        self.Ea_sei = nn.Parameter(torch.tensor(0.55), requires_grad=True)
        self.Ea_pl = nn.Parameter(torch.tensor(0.40), requires_grad=True)
        self.entangler_weights = nn.Parameter(torch.randn(entangler_layers, n_qubits) * 0.1)
        self.entangler_rzz = nn.Parameter(torch.randn(entangler_layers, n_qubits - 1) * 0.1)

        def _apply_rich_entangler(weights, rzz_w):
            for l in range(weights.shape[0]):
                for i in range(n_qubits):
                    qml.RY(weights[l, i], wires=i)
                for i in range(n_qubits - 1):
                    qml.IsingZZ(rzz_w[l, i], wires=[i, i + 1])

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def circuit(inputs, ew, erzz):
            qml.RY(inputs[:, 0], wires=0)
            qml.RY(inputs[:, 1], wires=1)
            qml.RY(inputs[:, 2], wires=2)
            qml.RY(inputs[:, 3], wires=3)
            _apply_rich_entangler(ew, erzz)
            return tuple(qml.expval(qml.PauliZ(i)) for i in range(n_qubits))

        self.circuit = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out_device, out_dtype = x.device, x.dtype
        voltage_angle = x[:, 0] * torch.pi
        current_angle = x[:, 1] * torch.pi
        time_angle = x[:, 3] * torch.pi
        temp_c = x[:, 2]
        temp_k = torch.clamp(temp_c + 273.15, min=1.0)
        inv_t = 1.0 / temp_k
        inv_t_ref = 1.0 / self.T_ref
        Ea_sei_actual = self.Ea_sei * 10000.0
        Ea_pl_actual = self.Ea_pl * 10000.0
        sei_term = torch.exp((Ea_sei_actual / self.R) * (inv_t_ref - inv_t))
        plating_term = torch.exp((Ea_pl_actual / self.R) * (inv_t - inv_t_ref))
        phi = sei_term + plating_term
        theta_temp = torch.pi * phi / 4.0
        angles = torch.stack([voltage_angle, current_angle, time_angle, theta_temp], dim=1)
        angles_cpu = angles.to("cpu")
        ew_cpu = self.entangler_weights.to("cpu")
        erzz_cpu = self.entangler_rzz.to("cpu")
        q_out = self.circuit(angles_cpu, ew_cpu, erzz_cpu)
        return torch.stack(q_out, dim=1).to(device=out_device, dtype=out_dtype)

class TEQTransformer(nn.Module):
    """Full TE-Q-Transformer Architecture for SOH Sequence Regression."""
    def __init__(self, cfg: Optional[TEQTransformerConfig] = None) -> None:
        super().__init__()
        self.cfg = cfg or TEQTransformerConfig()
        self.quantum_embed = QuantumEmbeddingLayer(
            n_qubits=self.cfg.quantum_dim,
            q_device=self.cfg.q_device,
            entangler_layers=self.cfg.entangler_layers,
        )
        self.quantum_proj = nn.Linear(self.cfg.quantum_dim, self.cfg.d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, self.cfg.d_model))
        self.pos_encoder = PositionalEncodingTEQ(self.cfg.d_model, max_len=self.cfg.seq_len + 2)
        self.temporal_smooth = nn.Conv1d(
            in_channels=self.cfg.d_model,
            out_channels=self.cfg.d_model,
            kernel_size=self.cfg.temporal_kernel_size,
            padding=self.cfg.temporal_kernel_size // 2,
            bias=False,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.d_model,
            nhead=self.cfg.n_heads,
            dim_feedforward=self.cfg.dim_feedforward,
            dropout=self.cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.cfg.n_layers)
        self.head = nn.Sequential(
            nn.Linear(self.cfg.d_model, self.cfg.head_hidden_dim),
            nn.GELU(),
            nn.Dropout(self.cfg.dropout),
            nn.Linear(self.cfg.head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        x_flat = x.reshape(batch_size * seq_len, 4)
        q_features = self.quantum_embed(x_flat)
        q_features = self.quantum_proj(q_features)
        q_sequence = q_features.reshape(batch_size, seq_len, self.cfg.d_model)
        q_sequence = self.temporal_smooth(q_sequence.transpose(1, 2)).transpose(1, 2)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        q_sequence = torch.cat([cls_tokens, q_sequence], dim=1)
        q_sequence = self.pos_encoder(q_sequence)
        encoded = self.transformer(q_sequence)
        pooled = encoded[:, 0, :]
        return self.head(pooled).squeeze(-1)

model = TEQTransformer().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print("=" * 70)
print(f"[Architecture] TE-Q-Transformer compiled successfully.")
print(f"[Architecture] Total Trainable Parameters: {total_params:,}")
print("=" * 70)


In [ ]:
# ==============================================================================
# 5. METRICS COMPUTATION & EVALUATION ROUTINE
# ==============================================================================
def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    """Computes RMSE, MAE, MAPE (%), R², and Max Error."""
    actual = np.asarray(actual, dtype=np.float64).reshape(-1)
    predicted = np.asarray(predicted, dtype=np.float64).reshape(-1)
    abs_error = np.abs(actual - predicted)
    denom = np.clip(np.abs(actual), a_min=1e-8, a_max=None)
    ss_res = np.sum((actual - predicted) ** 2)
    ss_tot = np.sum((actual - np.mean(actual)) ** 2)
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return {
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "MAE": float(mean_absolute_error(actual, predicted)),
        "MAPE (%)": float(np.mean(abs_error / denom) * 100.0),
        "R2": r2,
        "MaxE": float(np.max(abs_error)),
    }

def evaluate_model(model: nn.Module, test_loaders: Dict[str, DataLoader], test_cycles: Dict[str, np.ndarray], device: torch.device) -> Tuple[List[Dict[str, Any]], Dict[str, Dict[str, np.ndarray]]]:
    """Runs deterministic inference across all test partitions."""
    model.eval()
    records = []
    cell_arrays = {}
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            all_preds, all_trues = [], []
            for bx, by in loader:
                bx = bx.to(device)
                preds = model(bx).cpu().numpy().reshape(-1)
                all_preds.append(preds)
                all_trues.append(by.numpy().reshape(-1))
            y_pred = np.concatenate(all_preds)
            y_true = np.concatenate(all_trues)
            cycles = test_cycles[cell_id]
            cell_arrays[cell_id] = {"true": y_true, "pred": y_pred, "cycles": cycles}
            m = compute_metrics(y_true, y_pred)
            records.append({
                "cell": cell_id,
                "samples": len(y_true),
                "rmse": m["RMSE"],
                "mae": m["MAE"],
                "mape": m["MAPE (%)"],
                "r2": m["R2"],
                "max_error": m["MaxE"],
            })
    return records, cell_arrays


In [ ]:
# ==============================================================================
# 6. MODEL TRAINING / PRE-TRAINED CHECKPOINT LOADING
# ==============================================================================
# Check if pre-trained CALCE TE-Q weights are available
CKPT_CANDIDATES = [
    Path("calce/calce/extracted_teq/calce_results/ablation_study/01_rich_entangler/calce_teq_transformer_best.pth"),
    Path("../../calce/calce/extracted_teq/calce_results/ablation_study/01_rich_entangler/calce_teq_transformer_best.pth"),
    CKPT_DIR / "calce_teq_transformer_best.pth",
]

LOAD_EXISTING_CHECKPOINT = False
for p in CKPT_CANDIDATES:
    if p.exists():
        print(f"[Checkpoint] Found existing pre-trained checkpoint: {p.resolve()}")
        try:
            model.load_state_dict(torch.load(p, map_location=DEVICE))
            LOAD_EXISTING_CHECKPOINT = True
            print("[Checkpoint] Successfully loaded pre-trained TE-Q-Transformer weights.")
            break
        except Exception as e:
            print(f"[Checkpoint] Notice: Could not load {p}: {e}")

# If not loading or user requests retraining, train the model
FORCE_TRAIN = False  # Set to True to re-train even if checkpoint exists

if not LOAD_EXISTING_CHECKPOINT or FORCE_TRAIN:
    print("\n[Training] Starting TE-Q-Transformer training campaign on CALCE...")
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
    criterion = nn.MSELoss()
    
    best_loss = float("inf")
    patience_counter = 0
    max_epochs = 80
    patience = 20
    best_ckpt_path = CKPT_DIR / "calce_teq_transformer_best.pth"
    
    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        ep_t0 = time.time()
        model.train()
        losses = []
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())
        
        train_loss = float(np.mean(losses))
        scheduler.step(train_loss)
        
        if train_loss < best_loss:
            best_loss = train_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_ckpt_path)
            star = " * (best)"
        else:
            patience_counter += 1
            star = ""
            
        if epoch % 5 == 0 or epoch == 1 or star:
            print(f"  Epoch {epoch:02d}/{max_epochs:02d} | Train Loss: {train_loss:.6f} | Time: {time.time()-ep_t0:.2f}s{star}")
            
        if patience_counter >= patience:
            print(f"[Training] Early stopping triggered at epoch {epoch}.")
            break
            
    print(f"[Training] Completed in {time.time()-t0:.2f}s. Loading best weights...")
    model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))

# Run evaluation across held-out test splits
metric_records, cell_arrays = evaluate_model(model, test_loaders, test_cycles, DEVICE)
print("\n" + "=" * 70)
print("CALCE CROSS-CELL EVALUATION RESULTS:")
for r in metric_records:
    print(f"  Cell {r['cell']:12s} | Samples: {r['samples']:4d} | RMSE: {r['rmse']:.5f} | MAE: {r['mae']:.5f} | MAPE: {r['mape']:.2f}% | R2: {r['r2']:.4f} | MaxE: {r['max_error']:.4f}")
print("=" * 70)


In [ ]:
# ==============================================================================
# 7. PUBLICATION-QUALITY VISUALIZATION
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Panel 1: Held-Out Unseen Cell CS2_38 SOH Trajectory
ax1 = axes[0]
cs38_data = cell_arrays["CS2_38"]
ax1.plot(cs38_data["cycles"], cs38_data["true"], label="Ground Truth SOH", color="#1f77b4", linewidth=2.2)
ax1.plot(cs38_data["cycles"], cs38_data["pred"], label="TE-Q-Transformer (Ours)", color="#d62728", linestyle="--", linewidth=2.0)
ax1.set_title("Held-Out Unseen Cell CS2_38 (Cross-Cell)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Cycle Number", fontsize=11)
ax1.set_ylabel("Normalized State of Health (SOH)", fontsize=11)
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.legend(frameon=True, facecolor="white", framealpha=0.9)

# Panel 2: Temporal Extrapolation CS2_37_test Trajectory
ax2 = axes[1]
cs37_data = cell_arrays["CS2_37_test"]
ax2.plot(cs37_data["cycles"], cs37_data["true"], label="Ground Truth SOH", color="#1f77b4", linewidth=2.2)
ax2.plot(cs37_data["cycles"], cs37_data["pred"], label="TE-Q-Transformer (Ours)", color="#2ca02c", linestyle="--", linewidth=2.0)
ax2.set_title("Held-Out Extended Cycles CS2_37 (Temporal Extrapolation)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Cycle Number", fontsize=11)
ax2.set_ylabel("Normalized State of Health (SOH)", fontsize=11)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(frameon=True, facecolor="white", framealpha=0.9)

plt.tight_layout()
fig_traj_path = FIG_DIR / "calce_soh_trajectories.png"
plt.savefig(fig_traj_path, bbox_inches="tight")
plt.savefig(FIG_DIR / "calce_soh_trajectories.pdf", bbox_inches="tight")
plt.show()
print(f"[Figure] Saved SOH trajectory curves to {fig_traj_path}")

# Panel 3 & 4: Parity Scatter Plots
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

ax_p1 = axes2[0]
ax_p1.scatter(cs38_data["true"], cs38_data["pred"], color="#1f77b4", alpha=0.4, edgecolors="none", s=20)
min_val = min(cs38_data["true"].min(), cs38_data["pred"].min())
max_val = max(cs38_data["true"].max(), cs38_data["pred"].max())
ax_p1.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label="Ideal Parity (y = x)")
ax_p1.set_title("CS2_38 Parity Plot (Cross-Cell)", fontsize=12, fontweight="bold")
ax_p1.set_xlabel("Actual SOH", fontsize=11)
ax_p1.set_ylabel("Predicted SOH", fontsize=11)
ax_p1.grid(True, linestyle=":", alpha=0.6)
ax_p1.legend(frameon=True)

ax_p2 = axes2[1]
ax_p2.scatter(cs37_data["true"], cs37_data["pred"], color="#2ca02c", alpha=0.5, edgecolors="none", s=20)
min_val2 = min(cs37_data["true"].min(), cs37_data["pred"].min())
max_val2 = max(cs37_data["true"].max(), cs37_data["pred"].max())
ax_p2.plot([min_val2, max_val2], [min_val2, max_val2], 'k--', linewidth=1.5, label="Ideal Parity (y = x)")
ax_p2.set_title("CS2_37_test Parity Plot (Temporal Extrapolation)", fontsize=12, fontweight="bold")
ax_p2.set_xlabel("Actual SOH", fontsize=11)
ax_p2.set_ylabel("Predicted SOH", fontsize=11)
ax_p2.grid(True, linestyle=":", alpha=0.6)
ax_p2.legend(frameon=True)

plt.tight_layout()
fig_parity_path = FIG_DIR / "calce_parity_plots.png"
plt.savefig(fig_parity_path, bbox_inches="tight")
plt.savefig(FIG_DIR / "calce_parity_plots.pdf", bbox_inches="tight")
plt.show()
print(f"[Figure] Saved Parity plots to {fig_parity_path}")


In [ ]:
# ==============================================================================
# 8. METRICS EXPORT & SUMMARY TABLE
# ==============================================================================
df_results = pd.DataFrame(metric_records)
df_results["evaluation_type"] = [
    "Cross-Cell (Unseen Cell)" if c == "CS2_38" else "Temporal Extrapolation (30% Tail)"
    for c in df_results["cell"]
]

# Compute Macro Average across test cells
macro_row = pd.DataFrame([{
    "cell": "Macro Average",
    "samples": int(df_results["samples"].sum()),
    "rmse": float(df_results["rmse"].mean()),
    "mae": float(df_results["mae"].mean()),
    "mape": float(df_results["mape"].mean()),
    "r2": float(df_results["r2"].mean()),
    "max_error": float(df_results["max_error"].max()),
    "evaluation_type": "Overall Macro Summary",
}])

df_summary = pd.concat([df_results, macro_row], ignore_index=True)

# Save to CSV and JSON
csv_path = RESULTS_DIR / "calce_evaluation_metrics.csv"
json_path = RESULTS_DIR / "calce_evaluation_metrics.json"
df_summary.to_csv(csv_path, index=False)
df_summary.to_json(json_path, orient="records", indent=2)

print("=" * 75)
print("TE-Q-TRANSFORMER CALCE EVALUATION MASTER SUMMARY TABLE")
print("=" * 75)
display_cols = ["cell", "evaluation_type", "samples", "rmse", "mae", "mape", "r2", "max_error"]
print(df_summary[display_cols].to_string(index=False))
print("=" * 75)
print(f"[Export] Saved master metrics to {csv_path}")


In [ ]:
# ==============================================================================
# ARTIFACT PACKAGING FOR 1-CLICK KAGGLE DOWNLOAD
# ==============================================================================
import shutil
target_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
zip_base = str(target_dir / "CALCE_Evaluation_Results")
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"[SUCCESS] Saved 1-click download bundle to: {zip_base}.zip")
print("You can now download CALCE_Evaluation_Results.zip directly from the Kaggle Output viewer!")
